## Paso 0 — Cargamos las 18 tablas ya tipadas

Mismo patrón que los notebooks anteriores: leemos directo los Parquet del
notebook 00.


In [35]:
import pandas as pd
from pathlib import Path

RUTA_INTERIM = Path("../datos/interim")
ARCHIVOS_PARQUET = sorted(RUTA_INTERIM.glob("*.parquet"))
tablas = {archivo.stem: pd.read_parquet(archivo) for archivo in ARCHIVOS_PARQUET}
assert len(tablas) == 18, f"Se esperaban 18 tablas, se cargaron {len(tablas)}"
print(f"Tablas cargadas: {len(tablas)}")

Tablas cargadas: 18


## Paso 1 — ValorFlete: cada cotización, una por una (§6.1 del plan)

Con solo 48 líneas de cotización, mostramos cada valor real ordenado de
menor a mayor, en vez de resumir con un promedio — a este volumen,
resumir escondería más de lo que muestra.


In [36]:
cotizacion_lineas = tablas["cotizacion_lineas"]
print(f"Total de líneas de cotización: {len(cotizacion_lineas)}")
print()
print(cotizacion_lineas[["id", "fuente", "valorFlete"]].sort_values("valorFlete").to_string())
print()
print(cotizacion_lineas["valorFlete"].describe())


Total de líneas de cotización: 48

                                      id     fuente    valorFlete
36  B0F0A97D-A0CA-4EF9-8B01-9FA5F1A4DFE0     Manual  4.555500e+04
7   B316D6C3-5565-47B6-B19C-9F9CFC5850AA    Sicetac  1.564425e+06
28  EE9BBB4F-AA31-45E7-B400-578F49E641FC    Sicetac  1.564425e+06
19  74FFD733-53BD-491B-A921-5347C8EEE751    Sicetac  1.564425e+06
39  77F3030F-9CFF-4724-8151-31E5D867B645    Sicetac  1.564425e+06
1   7828D2DD-1C4B-4E11-BBFE-A1ED81FFAE99    Sicetac  1.802013e+06
37  EFCD2654-6355-40D3-9E97-613EA9C4D823    Sicetac  2.153978e+06
11  612415C9-9FBA-446E-AE45-E9B4EEFADB16  Tarifario  2.200000e+06
33  A835F194-5279-40FA-ACA1-DBA44EBC8B2F  Tarifario  2.200000e+06
20  AC3A2867-4808-48B6-A1D3-820274647E55  Tarifario  2.200000e+06
46  CE3FB119-6D2E-4F31-B547-F091C101F12A  Tarifario  2.200000e+06
41  53061505-363B-4B94-8F27-77907D6C1E8B  Tarifario  2.300000e+06
27  6949CFC1-9872-4378-8E91-4AC70F2BCB38  Tarifario  2.300000e+06
5   3FB01715-E528-4450-A4D2-00D1C5382F66 

In [37]:
manuales = cotizacion_lineas[cotizacion_lineas["fuente"] == "Manual"]
print(f"Líneas con Fuente=Manual (excluidas — no son precio real, puestas a mano por el usuario): {len(manuales)}")
print(manuales[["id", "valorFlete"]].to_string())

cotizacion_lineas_automatico = cotizacion_lineas[cotizacion_lineas["fuente"] != "Manual"]
print(f"\nQuedan {len(cotizacion_lineas_automatico)} de {len(cotizacion_lineas)} líneas (Fuente=Sicetac o Tarifario, calculadas por el sistema)")
print()
print(cotizacion_lineas_automatico["valorFlete"].describe())


Líneas con Fuente=Manual (excluidas — no son precio real, puestas a mano por el usuario): 3
                                      id    valorFlete
8   D8FA3F1E-643B-4192-9DFA-A622D2304036  6.788888e+06
35  9104DC9A-3F96-4275-B122-A1757C6F798E  4.500000e+09
36  B0F0A97D-A0CA-4EF9-8B01-9FA5F1A4DFE0  4.555500e+04

Quedan 45 de 48 líneas (Fuente=Sicetac o Tarifario, calculadas por el sistema)

count    4.500000e+01
mean     4.848106e+06
std      2.679450e+06
min      1.564425e+06
25%      2.300000e+06
50%      4.832788e+06
75%      7.667624e+06
max      8.650000e+06
Name: valorFlete, dtype: float64


## Paso 2 — ValorMercancia: valor asegurado de la carga (§6.1 del plan)

Distinto de ValorFlete (costo de transportar) — este es cuánto vale la
mercancía en sí. Mostramos cada valor real, buscando algo que se salga
de forma rara del resto, igual que con ValorFlete.


In [38]:
lineas_solicitud = tablas["lineas_solicitud"]
con_valor_mercancia = lineas_solicitud.dropna(subset=["valorMercancia"])
print(f"Líneas de solicitud con ValorMercancia: {len(con_valor_mercancia)} de {len(lineas_solicitud)}")
print()
print(con_valor_mercancia[["id", "valorMercancia"]].sort_values("valorMercancia").to_string())
print()
print(con_valor_mercancia["valorMercancia"].describe())


Líneas de solicitud con ValorMercancia: 38 de 43

                                      id  valorMercancia
3   AA7B215C-0FE1-46E9-A241-4EC3F4DF390B            12.0
25  0B78969D-AB82-4F65-922C-ED712455D83F            27.0
28  E237A3F0-8EC7-4CA2-8AFA-FC22A4F39FC6       1500000.0
30  0EF6C335-9DAE-4F61-BC5F-7DA445C78C78      12000000.0
20  C5C80238-6847-4055-A06F-BB470E4E9E22      22000000.0
12  F4C1CB7B-5CE4-49B2-AAC8-A27D2D0C46B9      22000000.0
33  CDE705A2-04FD-4503-ADBC-25F13872B726      22000000.0
15  C239BD84-BA13-4060-8A1B-002EE44AA592      30000000.0
4   1E7F4C45-8F8C-4039-8910-0121B58C4515      30000000.0
41  6F104278-EFED-487D-ADA3-CCFBB179860C      30000000.0
13  B9FDA6C1-D3D4-4312-B89F-BD0D4651AEAD      40000000.0
0   2AD1544F-48F0-418E-A5E1-61D95263C718      40000000.0
29  35BEB9C0-6497-41B8-9DA5-04E4B9170398      45000000.0
14  49073622-B9FB-4E9F-BDFA-D400B0EF0476      60000000.0
42  E385577F-A280-443C-B3C7-D50D3E58E136      60000000.0
16  3483D85D-B8A3-4952-8508-0D98C2B284

## Paso 3 — ValorAprovisionamiento: costo de reposicionar el contenedor vacío (§6.1 del plan)

Confirmado por el usuario que este campo se calcula automáticamente (no
es un valor puesto a mano) — a diferencia de ValorMercancia/Peso, aquí sí
podemos confiar en los valores tal cual.


In [39]:
con_valor_aprov = lineas_solicitud.dropna(subset=["valorAprovisionamiento"])
print(f"Líneas de solicitud con ValorAprovisionamiento: {len(con_valor_aprov)} de {len(lineas_solicitud)}")
print()
print(con_valor_aprov[["id", "valorAprovisionamiento"]].sort_values("valorAprovisionamiento").to_string())
print()
print(con_valor_aprov["valorAprovisionamiento"].describe())


Líneas de solicitud con ValorAprovisionamiento: 0 de 43

Empty DataFrame
Columns: [id, valorAprovisionamiento]
Index: []

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: valorAprovisionamiento, dtype: float64


## Paso 4 — Fuente: de dónde sacó CabinaFletes cada precio (§6.2 del plan)

La variable categórica más importante de esta fase, según el plan —
determina qué tan seguido se puede verificar un precio contra el mercado
externo (solo Sicetac/Tarifario se pueden verificar, Manual no).


In [40]:
print("Conteo por Fuente:")
print(cotizacion_lineas["fuente"].value_counts())
print()
print("Porcentaje por Fuente:")
print(cotizacion_lineas["fuente"].value_counts(normalize=True).mul(100).round(1))


Conteo por Fuente:
fuente
Tarifario    32
Sicetac      13
Manual        3
Name: count, dtype: int64

Porcentaje por Fuente:
fuente
Tarifario    66.7
Sicetac      27.1
Manual        6.2
Name: proportion, dtype: float64


## Paso 5 — Tipo, Modalidad, Estado de solicitudes (§6.2 del plan)

Tres categóricas chicas (13 filas). Ojo con Modalidad — ya sabemos que
está vacía en 12 de 13 filas (hueco conocido de la fase piloto, no un
hallazgo nuevo). Mostramos los nulos explícitamente (dropna=False) para
no escondзначerlos en el conteo.


In [41]:
solicitudes = tablas["solicitudes"]

for columna in ["tipo", "modalidad", "estado"]:
    print(f"--- {columna} ---")
    print(solicitudes[columna].value_counts(dropna=False))
    print()



--- tipo ---
tipo
Spot          8
Licitacion    5
Name: count, dtype: int64

--- modalidad ---
modalidad
NaN         12
Nacional     1
Name: count, dtype: int64

--- estado ---
estado
EnCosteo      6
Entregada     6
Descartada    1
Name: count, dtype: int64



## Paso 6 — Distribución de fechas, lado propio (§6.3 del plan)

Solo 13 solicitudes en una ventana de días — no da para hablar de
estacionalidad, solo confirmamos el rango exacto.


In [42]:
print("CreatedAt de las 13 solicitudes, por día:")
print(solicitudes["createdAt"].dt.date.value_counts().sort_index())
print()
print(f"Ventana: {solicitudes['createdAt'].min()} a {solicitudes['createdAt'].max()}")


CreatedAt de las 13 solicitudes, por día:
createdAt
2026-08-24    6
2026-08-25    2
2026-08-28    1
2026-08-31    4
Name: count, dtype: int64

Ventana: 2026-08-24 15:48:14 a 2026-08-31 17:25:40


## Paso 7 — VIGENCIA de tarifa-plaza, por mes (§6.3 del plan)

VIGENCIA es la fecha hasta la que una tarifa SAP es válida. Agrupamos
las 23.020 filas por mes para ver si hay picos de carga masiva de
tarifas nuevas — una investigación anterior (fuera de este EDA) había
visto picos en diciembre 2025 y julio 2026, sin confirmar con código
propio. Lo reproducimos aquí.


In [43]:
tarifa_plaza = tablas["tarifa-plaza-tblSAPTarifas-2022-09-01_a_hoy"]

vigencia_por_mes = tarifa_plaza["vigencia"].dt.to_period("M").value_counts().sort_index()
print(vigencia_por_mes.to_string())


vigencia
2024-01       8
2024-03       6
2024-04       6
2024-05      40
2024-06     101
2024-07       6
2024-08    8182
2024-09     307
2024-10     147
2024-11     145
2024-12     148
2025-01     216
2025-02     198
2025-03     231
2025-04     164
2025-05     374
2025-06     499
2025-07     397
2025-08     369
2025-09     641
2025-10     792
2025-11     709
2025-12    1625
2026-01     631
2026-02     713
2026-03     428
2026-04     705
2026-05     909
2026-06     740
2026-07    1681
2026-08     792
2026-09     939
2026-10     167
2026-11       1
2026-12       1
3999-12       2
Freq: M


## Paso 8 — Redundancia código↔texto en SAP (§6.4 del plan)

TIPOVEHICULO/VehiculoDescripcion y LINEANEGOCIO/LineaNegocioDescripcion
son pares código-texto que deberían ser 1 a 1 (mismo código, siempre la
misma descripción). En el EDA-1 un par similar sí tuvo discrepancias
reales — lo confirmamos aquí con código, no se asume.


In [44]:
for codigo, texto in [("tipovehiculo", "vehiculoDescripcion"), ("lineanegocio", "lineaNegocioDescripcion")]:
    mapeo = tarifa_plaza.groupby(codigo)[texto].nunique()
    inconsistentes = mapeo[mapeo > 1]
    print(f"{codigo} -> {texto}: {len(mapeo)} códigos únicos, {len(inconsistentes)} con más de una descripción distinta")
    if len(inconsistentes) > 0:
        print(inconsistentes)
    print()


tipovehiculo -> vehiculoDescripcion: 7 códigos únicos, 0 con más de una descripción distinta

lineanegocio -> lineaNegocioDescripcion: 5 códigos únicos, 0 con más de una descripción distinta



## Síntesis del notebook 04 — Análisis univariado (Fase 5, §6 del plan)

**Objetivo cumplido:** conocer la forma de las variables numéricas,
categóricas y de fecha más relevantes, tabla por tabla.

**Numéricas — lado propio (Paso 1-3):**
- `ValorFlete` (cotizacion_lineas): 3 de 48 líneas son `Fuente=Manual` —
  confirmado por el usuario como valores puestos al azar durante
  pruebas (no es experto en precios de flete), sin relación con la
  realidad — incluye un caso de $4.500 millones y otro de $45.555.
  Excluidas del análisis. Las 45 líneas restantes (Sicetac/Tarifario,
  calculadas por el sistema) dan una distribución sensata: mediana
  $4.832.788, rango $1.564.425-$8.650.000.
- `ValorMercancia`/`Peso` (lineas_solicitud): el usuario confirmó que
  también tienen valores puestos al azar, sin una columna tipo `Fuente`
  que permita separar cuáles son ruido — se declara la limitación
  completa de estas 2 columnas, no se intenta limpiar con precisión.
- `ValorAprovisionamiento`/`ValorRetornoContenedor` (lineas_solicitud y
  cotizacion_lineas): 100% vacías en las 91 filas combinadas. Investigado
  a fondo: la única línea que cumplía la condición de negocio para
  activarse (`RetornaContenedor=True`) usa la ruta de prueba malformada
  `"222"/"222"` — no hay tarifa que calcular para una ciudad que no
  existe. Explicado por el mismo ruido de datos de prueba ya conocido,
  no es un bug del sistema.

**Categóricas (Paso 4-5):**
- `Fuente` (cotizacion_lineas, la más importante de la fase): 66,7%
  Tarifario, 27,1% Sicetac, 6,2% Manual — ninguna línea usó `Historica`.
  Coherente con que Tarifario es la fuente ya reconfirmada al 100% contra
  SAP en el notebook 03.
- `Tipo`/`Estado` (solicitudes): razonablemente repartidas (8 Spot/5
  Licitación; 6 EnCosteo/6 Entregada/1 Descartada). `Modalidad`: 12 de 13
  vacías (hueco de piloto ya conocido, no nuevo).

**Fecha (Paso 6-7):**
- Lado propio: 13 solicitudes en solo 4 días distintos (24, 25, 28 y 31
  de agosto 2026) — confirma la ventana ya conocida, no da para
  estacionalidad.
- `VIGENCIA` de `tarifa-plaza` (23.020 filas, mes a mes): pico enorme en
  agosto 2024 (8.182 filas, ~5x el siguiente pico) — consistente con la
  carga inicial del sistema, no un patrón de negocio. Picos más chicos
  en diciembre 2025 (1.625) y julio 2026 (1.681), coherentes con una
  investigación anterior (decreto de salario mínimo para diciembre;
  julio sin explicación confirmada). 2 filas con `VIGENCIA=3999-12` — un
  valor centinela de "nunca vence", mismo patrón que `VALID_TO=9999` del
  catálogo de clientes.

**Redundancia (Paso 8):** `TIPOVEHICULO`/`VehiculoDescripcion` (7
códigos) y `LINEANEGOCIO`/`LineaNegocioDescripcion` (5 códigos) son
perfectamente 1 a 1 en SAP — sin discrepancias, a diferencia de un caso
similar en el EDA-1.

**Qué le queda listo al siguiente notebook:** entendimiento de la forma
de cada variable relevante, con las limitaciones de confiabilidad del
lado propio ya declaradas explícitamente. El siguiente paso del plan es
la Fase 6 (Bivariado/multivariado, §7) — el corazón de esta HU, donde se
responde qué tan cerca cotiza CabinaFletes del piso/plaza de mercado.
